In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("../data/processed/cleaned_arabam.csv")
print(f"Dataset Ready: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset Ready: 1000 rows, 18 columns


# 1. Feature-Target Separation and Train-Test Split
Splitting the dataset into feature matrix (X) and target variable (y). Applying 80/20 train-test split with fixed random seed for reproducibility.

In [2]:
X = df.drop(columns=["price", "series", "model"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train Shape: {X_train.shape} | y_train Shape: {y_train.shape}")
print(f"X_test Shape: {X_test.shape} | y_test Shape: {y_test.shape}")

X_train Shape: (800, 15) | y_train Shape: (800,)
X_test Shape: (200, 15) | y_test Shape: (200,)


# 2. Preprocessing Pipeline (ColumnTransformer)
Building automated imputation, scaling, and categorical encoding pipelines. 
- Numerical features: Median Imputation + StandardScaler
- Categorical features: Most Frequent Imputation + OneHotEncoder (handle_unknown='ignore')

In [3]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical Features ({len(num_features)}): {num_features}")
print(f"Categorical Features ({len(cat_features)}): {cat_features}")

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

print("\n[SUCCESS] Preprocessor ColumnTransformer successfully constructed.")

Numerical Features (9): ['year', 'kilometer', 'engine_displacement_cc', 'engine_power_hp', 'avg_fuel_consumption', 'painted_count', 'local_painted_count', 'changed_count', 'is_original']
Categorical Features (6): ['brand', 'transmission', 'fuel_type', 'body_type', 'color', 'drivetrain']

[SUCCESS] Preprocessor ColumnTransformer successfully constructed.


# 3. Model Training & Baseline Benchmarking
Evaluating regression algorithms with TransformedTargetRegressor (log1p/expm1):
1. Ridge Regression (Linear Regularized Baseline)
2. Random Forest Regressor (Non-linear Bagging Ensemble)
Evaluation metrics: R2 Score, RMSE, and MAE (TRY).

In [4]:

from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

models = {
    "Ridge Regression" : Ridge(alpha=1),
    "Random Forest" : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []

for model_name, model_instance in models.items():
    
    raw_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model_instance)
    ])

    model_pipeline = TransformedTargetRegressor(
        regressor=raw_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results.append({
        "Model" : model_name,
        "R2 Score" : round(r2, 4),
        "RMSE (TRY)" : f"{rmse:,.0f} TL",
        "MAE (TRY)" : f"{mae:,.0f} TL" 
    })

    results_df = pd.DataFrame(results)
    display(results_df)

,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,Ridge Regression,0.6567,"452,976 TL","178,344 TL"


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,Ridge Regression,0.6567,"452,976 TL","178,344 TL"
1,Random Forest,0.6528,"455,507 TL","178,688 TL"


> **Baseline Benchmark Insights:**
> - Both regularized linear regression (Ridge: $R^2 \approx 0.657$) and bagging ensemble (Random Forest: $R^2 \approx 0.653$) yield comparable baseline scores.
> - Average absolute error (MAE) is around **178K TRY** across unseen test samples.
> - Next step: Evaluate gradient boosting algorithms (LightGBM / XGBoost) and tune hyperparameters to capture non-linear feature interactions and push $R^2$ beyond 0.70+.

# 4. Gradient Boosting Evaluation (LightGBM Benchmark)
Introducing LightGBM Regressor into the evaluation loop alongside Ridge and Random Forest to benchmark iterative boosting performance against bagging and linear baselines.

In [5]:
from lightgbm import LGBMRegressor

benchmark_models = {
    "Ridge Regression" : Ridge(alpha=1),
    "Random Forest" : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "LightGBM" : LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1)
}

comparison_results = []

for model_name, model_inst in benchmark_models.items():

    raw_pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model_inst)
    ])

    transformed_model = TransformedTargetRegressor(
        regressor=raw_pipe,
        func=np.log1p,
        inverse_func=np.expm1
    )

    transformed_model.fit(X_train, y_train)
    y_preds = transformed_model.predict(X_test)

    r2 = r2_score(y_test, y_preds)
    rmse = np.sqrt(mean_squared_error(y_test, y_preds))
    mae = mean_absolute_error(y_test, y_preds)

    comparison_results.append({
        "Model" : model_name,
        "R2 Score" : round(r2, 4),
        "RMSE (TRY)" : f"{rmse:,.0f} TL",
        "MAE (TRY)" : f"{mae:,.0f} TL"
    })

    comparison_df = pd.DataFrame(comparison_results)
    display(comparison_df)

,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,Ridge Regression,0.6567,"452,976 TL","178,344 TL"


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,Ridge Regression,0.6567,"452,976 TL","178,344 TL"
1,Random Forest,0.6528,"455,507 TL","178,688 TL"


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,Ridge Regression,0.6567,"452,976 TL","178,344 TL"
1,Random Forest,0.6528,"455,507 TL","178,688 TL"
2,LightGBM,0.7054,"419,613 TL","162,088 TL"


> **Benchmark Insights:**
> - **LightGBM Outperforms:** Gradient boosting delivers a noticeable performance boost over bagging and linear baselines, pushing $R^2$ from **~0.65 to 0.7054** and reducing MAE down to **162,088 TRY**.
> - **Error Reduction:** Sequential gradient boosting effectively handles non-linear interactions between brand, year, and mileage, trimming test RMSE by over **33K TRY**.
> - **Next Objective:** Incorporate high-cardinality features (`series` and `model`) via Target Encoding and engineer domain-specific signals (`vehicle_age`, `km_per_year`) to target $R^2 \ge 0.75+$.

# 5. Feature Engineering (Domain-Specific Indicators)
Deriving domain-specific indicators to capture vehicle wear and usage intensity:
- `vehicle_age`: Calculated relative to the current year (2026 - year).
- `km_per_year`: Annual mileage usage intensity ($\frac{\text{kilometer}}{\text{vehicle\_age} + 1}$).

In [6]:
df_fe = df.copy()

current_year = 2026
df_fe["vehicle_age"] = current_year - df_fe["year"]
df_fe["km_per_year"] = df_fe["kilometer"] / (df_fe["vehicle_age"] + 1)

X_fe = df_fe.drop(columns=["price", "series", "model"])
y_fe = df_fe["price"]

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, test_size=0.2, random_state=42)

num_features_fe = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_features_fe = X_train_fe.select_dtypes(include=["object"]).columns.tolist()

print(f"Updated Numerical Features ({len(num_features_fe)}): {num_features_fe}")

preprocessor_fe = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_features_fe),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_features_fe)
])

lgbm_fe_pipeline = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", preprocessor_fe),
        ("regressor", LGBMRegressor(n_estimators=150, learning_rate=0.5, random_state=42, verbose=-1))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

lgbm_fe_pipeline.fit(X_train_fe, y_train_fe)
y_pred_fe = lgbm_fe_pipeline.predict(X_test_fe)

r2_fe = r2_score(y_test_fe, y_pred_fe)
rmse_fe = np.sqrt(mean_squared_error(y_test_fe, y_pred_fe))
mae_fe = mean_absolute_error(y_test_fe, y_pred_fe)

fe_result = pd.DataFrame([{
    "Model" : "LightGBM (With Feature Engineering)",
    "R2 Score" : round(r2_fe, 4),
    "RMSE (TRY)" : f"{rmse_fe:,.0f} TL",
    "MAE (TRY)" : f"{mae_fe:,.0f} TL"
}])

display(fe_result)

Updated Numerical Features (11): ['year', 'kilometer', 'engine_displacement_cc', 'engine_power_hp', 'avg_fuel_consumption', 'painted_count', 'local_painted_count', 'changed_count', 'is_original', 'vehicle_age', 'km_per_year']


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,LightGBM (With Feature Engineering),0.7006,"423,035 TL","175,145 TL"


# 6. High-Cardinality Feature Integration (Target Encoding for 'series')
Integrating the vehicle series feature using Out-of-Fold Target Encoding to capture model-tier price variance without exploding dimensionality.

In [7]:
from sklearn.preprocessing import TargetEncoder

X_te = df.drop(columns=["price", "model"])
y_te = df["price"]

X_train_te, X_test_te, y_train_te, y_test_te = train_test_split(X_te, y_te, test_size=0.2, random_state=42)

num_cols = X_train_te.select_dtypes(include=[np.number]).columns.tolist()
high_card_col = ["series"]
low_card_cols = [c for c in X_train_te.select_dtypes(include=["object"]).columns if c != "series"]

print(f"Numerical ({len(num_cols)}): {num_cols}")
print(f"Target Encoded High-Cardinality (1): {high_card_col}")
print(f"One-Hot Encoded Low-Cardinality ({len(low_card_cols)}): {low_card_cols}")

preprocessor_te = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("target_enc", TargetEncoder(cv=5, smooth="auto", random_state=42), high_card_col),
    ("onehot", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), low_card_cols)
])

lgbm_te_pipeline = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", preprocessor_te),
        ("regressor", LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

lgbm_te_pipeline.fit(X_train_te, y_train_te)
y_pred_te = lgbm_te_pipeline.predict(X_test_te)

r2_te = r2_score(y_test_te, y_pred_te)
rmse_te = np.sqrt(mean_squared_error(y_test_te, y_pred_te))
mae_te = mean_absolute_error(y_test_te, y_pred_te)

te_result = pd.DataFrame([{
    "Model" : "LightGBM (With Target Encoded 'series')",
    "R2 Score" : round(r2_te, 4),
    "RMSE (TRY)" : f"{rmse_te:,.0f} TL",
    "MAE (TRY)" : f"{mae_te:,.0f} TL"
}])

display(te_result)

Numerical (9): ['year', 'kilometer', 'engine_displacement_cc', 'engine_power_hp', 'avg_fuel_consumption', 'painted_count', 'local_painted_count', 'changed_count', 'is_original']
Target Encoded High-Cardinality (1): ['series']
One-Hot Encoded Low-Cardinality (6): ['brand', 'transmission', 'fuel_type', 'body_type', 'color', 'drivetrain']


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,LightGBM (With Target Encoded 'series'),0.7224,"407,316 TL","145,481 TL"


# 7. Hyperparameter Tuning (GridSearchCV on LightGBM)
Systematically tuning tree depth, leaf capacity, and learning rates with 5-fold cross-validation to maximize generalization and finalize our production model parameters.

In [8]:
from sklearn.model_selection import GridSearchCV

base_pipe = Pipeline([
    ("preprocessor", preprocessor_te),
    ("regressor", LGBMRegressor(random_state=42, verbose=-1))
])

tuned_target_model = TransformedTargetRegressor(
    regressor=base_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

param_grid = {
    "regressor__regressor__n_estimators" : [100, 150, 200],
    "regressor__regressor__learning_rate" : [0.03, 0.05, 0.08],
    "regressor__regressor__num_leaves" : [15, 31],
    "regressor__regressor__max_depth" : [-1, 5, 8]
}

print("[INFO] Starting Grid Search across parameter grid...")
grid_search = GridSearchCV(
    estimator=tuned_target_model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_te, y_train_te)

print("\n--- Best Hyperparameters Found ---")
for param, val in grid_search.best_params_.items():
    clean_name = param.replace("regressor__regressor__", "")
    print(f"{clean_name:<20}: {val}")

best_model = grid_search.best_estimator_
y_tuned_preds = best_model.predict(X_test_te)

r2_tuned = r2_score(y_test_te, y_tuned_preds)
rmse_tuned = np.sqrt(mean_squared_error(y_test_te, y_tuned_preds))
mae_tuned = mean_absolute_error(y_test_te, y_tuned_preds)

tuned_summary = pd.DataFrame([{
    "Model" : "LightGBM (GridSearchCV Tuned)",
    "R2 Score" : round(r2_tuned, 4),
    "RMSE (TRY)" : f"{rmse_tuned:,.0f} TL",
    "MAE (TRY)" : f"{mae_tuned:,.0f} TL"
}])

display(tuned_summary)

[INFO] Starting Grid Search across parameter grid...
Fitting 5 folds for each of 54 candidates, totalling 270 fits

--- Best Hyperparameters Found ---
learning_rate       : 0.08
max_depth           : -1
n_estimators        : 150
num_leaves          : 15


,Model,R2 Score,RMSE (TRY),MAE (TRY)
0,LightGBM (GridSearchCV Tuned),0.7501,"386,468 TL","135,197 TL"


> **Final Model Optimization Insights:**
> - **Target Goal Reached ($R^2 \ge 0.75$):** Hyperparameter tuning via 5-fold cross-validation boosted the test $R^2$ score to **0.7501**.
> - **Error Reduction:** Reduced Mean Absolute Error (MAE) to **135,197 TRY** (a cumulative ~43K TRY improvement over the initial baseline model)[cite: 8].
> - **Optimal Hyperparameters:** `learning_rate=0.08`, `n_estimators=150`, `num_leaves=15`, `max_depth=-1`.
> - **Conclusion:** The pipeline is fully optimized and ready to be refactored into production scripts (`src/train.py`) and saved as a joblib artifact (`models/best_car_price_model.joblib`).